# Per-User, Per-Period MAPE Evaluation

Evaluates all 5 trained models (AutoARIMA, AutoETS, SARIMAX, Prophet, iTransformer)  
for every user across the 4 test periods defined in `evaluation_protocol.json`.

**Outputs:**
- `artifacts/clustering/per_user_period_mape.parquet` — columns: `user_id`, `model`, `period_id`, `MAPE`
- `artifacts/clustering/cluster_period_mape.parquet` — columns: `cluster_key`, `model`, `period_id`, `cluster_MAPE`

Run from the **project root** with the project virtual environment active.

In [1]:
import io, json, math, warnings, zipfile, tempfile
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

ROOT       = Path('').resolve().parents[1]   # project root (two levels up from src/evaluation/)
MODEL_DIR  = ROOT / 'model_weights'
ARTIFACTS  = ROOT / 'artifacts' / 'clustering'

TRAIN_PATH    = ROOT / 'master_long_hourly_train_2012_2013.csv'
VAL_PATH      = ROOT / 'master_long_hourly_validation_2014_01_04.csv'
TEST_PATH     = ROOT / 'master_long_hourly_test_2014_05_12.csv'
CALENDAR_PATH = ROOT / 'calendar_features_hourly.csv'
MAPPING_PATH  = ARTIFACTS / 'user_cluster_mapping.csv'
PROTOCOL_PATH = ARTIFACTS / 'evaluation_protocol.json'

EXOG_COLS     = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend', 'month_sin', 'month_cos']
CHUNK_H       = 720   # iTransformer rolling horizon
LOOKBACK      = 672   # 4-week lookback for AutoARIMA/AutoETS

print('ROOT:', ROOT)

ROOT: /Users/jackchiu/assignments/Forecasting/IEOR4578_Electricity_Project


## 1. Load Data

In [2]:
calendar = pd.read_csv(CALENDAR_PATH, parse_dates=['timestamp'])
calendar = calendar.rename(columns={'timestamp': 'ds'})

def load_long(path):
    df = pd.read_csv(path, parse_dates=['timestamp'])
    df = df.rename(columns={'client_id': 'unique_id', 'timestamp': 'ds'})
    df = df.merge(calendar[['ds'] + EXOG_COLS], on='ds', how='left')
    return df.sort_values(['unique_id', 'ds']).reset_index(drop=True)

train = load_long(TRAIN_PATH)
val   = load_long(VAL_PATH)
test  = load_long(TEST_PATH)

print(f'train : {train.shape}  {train["ds"].min()} -> {train["ds"].max()}')
print(f'val   : {val.shape}    {val["ds"].min()} -> {val["ds"].max()}')
print(f'test  : {test.shape}   {test["ds"].min()} -> {test["ds"].max()}')

train : (2721888, 10)  2012-01-01 00:00:00 -> 2013-12-31 23:00:00
val   : (445536, 10)    2014-01-01 00:00:00 -> 2014-04-30 23:00:00
test  : (913536, 10)   2014-05-01 00:00:00 -> 2014-12-31 23:00:00


## 2. Cluster Mapping & Scale Factors

In [3]:
mapping       = pd.read_csv(MAPPING_PATH)
cluster_map   = mapping[mapping['cluster_id'] != -1].set_index('user_id')['cluster_id']
outlier_users = mapping[mapping['cluster_id'] == -1]['user_id'].tolist()
all_users     = mapping['user_id'].tolist()

# Scale factor: user_train_mean / cluster_train_mean
user_means    = train.groupby('unique_id')['y'].mean()
scale_factors = {}
for cid, users in cluster_map.groupby(cluster_map):
    cm = user_means[users.index.tolist()].mean()
    for u in users.index:
        scale_factors[u] = float(user_means[u] / cm)
for u in outlier_users:
    scale_factors[u] = 1.0

print(f'Cluster 0: {(cluster_map == 0).sum()} users')
print(f'Cluster 1: {(cluster_map == 1).sum()} users')
print(f'Outliers : {len(outlier_users)} users')

Cluster 0: 84 users
Cluster 1: 64 users
Outliers : 8 users


## 3. Load Evaluation Protocol (Test Periods)

In [4]:
with open(PROTOCOL_PATH) as f:
    protocol = json.load(f)

periods = protocol['equal_size_test_periods']
for p in periods:
    print(f"Period {p['period_id']}: {p['start']} -> {p['end']}  ({p['n_hours']} hours)")

Period 1: 2014-05-01 00:00:00 -> 2014-06-30 23:00:00  (1464 hours)
Period 2: 2014-07-01 00:00:00 -> 2014-08-30 23:00:00  (1464 hours)
Period 3: 2014-08-31 00:00:00 -> 2014-10-31 23:00:00  (1464 hours)
Period 4: 2014-11-01 00:00:00 -> 2014-12-31 23:00:00  (1464 hours)


## 4. Utilities

In [5]:
def mape(y, yhat):
    """MAPE on 0-100 scale; skips zero / NaN actuals (per evaluation_protocol.json)."""
    y, yhat = np.asarray(y, float), np.asarray(yhat, float)
    pos = (y > 0) & np.isfinite(y) & np.isfinite(yhat)
    return float(np.mean(np.abs((y[pos] - yhat[pos]) / y[pos])) * 100) if pos.sum() > 0 else float('nan')

def load_from_zip(zip_name, member_path):
    with zipfile.ZipFile(MODEL_DIR / zip_name) as z:
        with z.open(member_path) as f:
            return joblib.load(io.BytesIO(f.read()))

def compute_per_user_period_mape(preds_df, pred_col, model_name):
    """
    Given a predictions DataFrame with [unique_id, ds, <pred_col>],
    merge with test actuals, compute per-user MAPE for each period.
    Returns DataFrame: [user_id, model, period_id, MAPE]
    """
    preds_df['ds'] = pd.to_datetime(preds_df['ds'])
    merged = test[['unique_id', 'ds', 'y']].merge(
        preds_df[['unique_id', 'ds', pred_col]], on=['unique_id', 'ds'], how='inner'
    )
    rows = []
    for p in periods:
        p_start = pd.Timestamp(p['start'])
        p_end   = pd.Timestamp(p['end']) + pd.Timedelta(hours=23)
        mask    = (merged['ds'] >= p_start) & (merged['ds'] <= p_end)
        p_df    = merged[mask]
        for uid, grp in p_df.groupby('unique_id'):
            rows.append({
                'user_id':   uid,
                'model':     model_name,
                'period_id': p['period_id'],
                'MAPE':      mape(grp['y'].values, grp[pred_col].values),
            })
    return pd.DataFrame(rows)

print('Utilities ready.')

Utilities ready.


## 5. AutoARIMA

In [6]:
test_h = test['ds'].nunique()
all_preds_autoarima = []

for cid in [0, 1]:
    sf = load_from_zip('AutoARIMA_models.zip', f'models/autoarima_final_cluster_{cid}.joblib')
    p  = sf.predict(h=test_h).reset_index(drop=True)
    col = [c for c in p.columns if c not in ('unique_id', 'ds')][0]
    for user in cluster_map[cluster_map == cid].index:
        udf = p[['ds', col]].copy()
        udf['unique_id'] = user
        udf[col] = udf[col] * scale_factors.get(user, 1.0)
        all_preds_autoarima.append(udf.rename(columns={col: 'AutoARIMA'})[['unique_id', 'ds', 'AutoARIMA']])

for user in outlier_users:
    sf = load_from_zip('AutoARIMA_models.zip', f'models/autoarima_final_{user}.joblib')
    p  = sf.predict(h=test_h).reset_index(drop=True)
    col = [c for c in p.columns if c not in ('unique_id', 'ds')][0]
    udf = p[['ds', col]].copy()
    udf['unique_id'] = user
    all_preds_autoarima.append(udf.rename(columns={col: 'AutoARIMA'})[['unique_id', 'ds', 'AutoARIMA']])

autoarima_preds = pd.concat(all_preds_autoarima, ignore_index=True)
mape_autoarima  = compute_per_user_period_mape(autoarima_preds, 'AutoARIMA', 'AutoARIMA')
print('AutoARIMA done. Sample:')
print(mape_autoarima.groupby('period_id')['MAPE'].mean().round(2))

AutoARIMA done. Sample:
period_id
1    13.65
2    15.92
3    14.86
4    16.22
Name: MAPE, dtype: float64


## 6. AutoETS

In [7]:
all_preds_autoets = []

for cid in [0, 1]:
    sf  = load_from_zip('AutoETS_models.zip', f'models/autoets_final_cluster_{cid}.joblib')
    p   = sf.predict(h=test_h).reset_index(drop=True)
    col = [c for c in p.columns if c not in ('unique_id', 'ds')][0]
    for user in cluster_map[cluster_map == cid].index:
        udf = p[['ds', col]].copy()
        udf['unique_id'] = user
        udf[col] = udf[col] * scale_factors.get(user, 1.0)
        all_preds_autoets.append(udf.rename(columns={col: 'AutoETS'})[['unique_id', 'ds', 'AutoETS']])

for user in outlier_users:
    sf  = load_from_zip('AutoETS_models.zip', f'models/autoets_final_{user}.joblib')
    p   = sf.predict(h=test_h).reset_index(drop=True)
    col = [c for c in p.columns if c not in ('unique_id', 'ds')][0]
    udf = p[['ds', col]].copy()
    udf['unique_id'] = user
    all_preds_autoets.append(udf.rename(columns={col: 'AutoETS'})[['unique_id', 'ds', 'AutoETS']])

autoets_preds = pd.concat(all_preds_autoets, ignore_index=True)
mape_autoets  = compute_per_user_period_mape(autoets_preds, 'AutoETS', 'AutoETS')
print('AutoETS done. Sample:')
print(mape_autoets.groupby('period_id')['MAPE'].mean().round(2))

AutoETS done. Sample:
period_id
1    21.18
2    31.78
3    38.71
4    52.82
Name: MAPE, dtype: float64


## 7. SARIMAX

In [8]:
ts_df = (
    test[['ds'] + EXOG_COLS]
    .drop_duplicates('ds')
    .sort_values('ds')
    .reset_index(drop=True)
)

all_preds_sarimax = []

for cid in [0, 1]:
    cluster_uid = f'cluster_{cid}'
    sf  = load_from_zip('SARIMAX_models.zip', f'models/sarimax_final_{cluster_uid}.joblib')
    X   = ts_df.copy()
    X['unique_id'] = cluster_uid
    p   = sf.predict(h=test_h, X_df=X[['unique_id', 'ds'] + EXOG_COLS]).reset_index(drop=True)
    col = [c for c in p.columns if c not in ('unique_id', 'ds')][0]
    for user in cluster_map[cluster_map == cid].index:
        udf = p[['ds', col]].copy()
        udf['unique_id'] = user
        udf[col] = udf[col] * scale_factors.get(user, 1.0)
        all_preds_sarimax.append(udf.rename(columns={col: 'SARIMAX'})[['unique_id', 'ds', 'SARIMAX']])

for user in outlier_users:
    sf  = load_from_zip('SARIMAX_models.zip', f'models/sarimax_final_{user}.joblib')
    X   = ts_df.copy()
    X['unique_id'] = user
    p   = sf.predict(h=test_h, X_df=X[['unique_id', 'ds'] + EXOG_COLS]).reset_index(drop=True)
    col = [c for c in p.columns if c not in ('unique_id', 'ds')][0]
    udf = p[['ds', col]].copy()
    udf['unique_id'] = user
    all_preds_sarimax.append(udf.rename(columns={col: 'SARIMAX'})[['unique_id', 'ds', 'SARIMAX']])

sarimax_preds = pd.concat(all_preds_sarimax, ignore_index=True)
mape_sarimax  = compute_per_user_period_mape(sarimax_preds, 'SARIMAX', 'SARIMAX')
print('SARIMAX done. Sample:')
print(mape_sarimax.groupby('period_id')['MAPE'].mean().round(2))

SARIMAX done. Sample:
period_id
1    14.64
2    18.34
3    18.38
4    19.40
Name: MAPE, dtype: float64


## 8. Prophet

In [9]:
future_base = (
    test[['ds', 'is_weekend']]
    .drop_duplicates('ds')
    .sort_values('ds')
    .reset_index(drop=True)
)

all_preds_prophet = []

for cid in [0, 1]:
    cluster_uid = f'cluster_{cid}'
    m  = load_from_zip('Prophet_models.zip', f'models/prophet_final_{cluster_uid}.joblib')
    fc = m.predict(future_base.copy())
    for user in cluster_map[cluster_map == cid].index:
        all_preds_prophet.append(pd.DataFrame({
            'unique_id': user,
            'ds':        fc['ds'].values,
            'Prophet':   fc['yhat'].values * scale_factors.get(user, 1.0),
        }))

for user in outlier_users:
    m  = load_from_zip('Prophet_models.zip', f'models/prophet_final_{user}.joblib')
    fc = m.predict(future_base.copy())
    all_preds_prophet.append(pd.DataFrame({
        'unique_id': user,
        'ds':        fc['ds'].values,
        'Prophet':   fc['yhat'].values,
    }))

prophet_preds = pd.concat(all_preds_prophet, ignore_index=True)
mape_prophet  = compute_per_user_period_mape(prophet_preds, 'Prophet', 'Prophet')
print('Prophet done. Sample:')
print(mape_prophet.groupby('period_id')['MAPE'].mean().round(2))

Prophet done. Sample:
period_id
1    16.52
2    22.02
3    18.42
4    22.26
Name: MAPE, dtype: float64


## 9. iTransformer (Rolling)

In [10]:
from neuralforecast import NeuralForecast

def add_calendar_features(df):
    """Compute calendar exogenous features from ds column."""
    df = df.copy()
    df['hour_sin']   = np.sin(2 * np.pi * df['ds'].dt.hour / 24)
    df['hour_cos']   = np.cos(2 * np.pi * df['ds'].dt.hour / 24)
    df['dow_sin']    = np.sin(2 * np.pi * df['ds'].dt.dayofweek / 7)
    df['dow_cos']    = np.cos(2 * np.pi * df['ds'].dt.dayofweek / 7)
    df['is_weekend'] = (df['ds'].dt.dayofweek >= 5).astype(int)
    df['month_sin']  = np.sin(2 * np.pi * df['ds'].dt.month / 12)
    df['month_cos']  = np.cos(2 * np.pi * df['ds'].dt.month / 12)
    return df

def get_itransformer_dir(cluster_key):
    tmp    = tempfile.mkdtemp()
    prefix = f'itransformer_final_{cluster_key}/'
    with zipfile.ZipFile(MODEL_DIR / 'itransformer_models.zip') as z:
        for member in z.namelist():
            if member.startswith(prefix):
                z.extract(member, tmp)
    return str(Path(tmp) / f'itransformer_final_{cluster_key}')

train_val  = pd.concat([train, val], ignore_index=True).sort_values(['unique_id', 'ds']).reset_index(drop=True)
test_dates = sorted(test['ds'].unique())
n_chunks   = math.ceil(len(test_dates) / CHUNK_H)
print(f'Rolling prediction: {n_chunks} chunks of up to {CHUNK_H} hours each')

cluster_groups = {
    'cluster_0': cluster_map[cluster_map == 0].index.tolist(),
    'cluster_1': cluster_map[cluster_map == 1].index.tolist(),
    'outliers':  outlier_users,
}

all_preds_itransformer = []

for cluster_key, users in cluster_groups.items():
    print(f'\n--- iTransformer: {cluster_key} ({len(users)} users) ---')
    nf      = NeuralForecast.load(get_itransformer_dir(cluster_key))
    history = train_val[train_val['unique_id'].isin(users)].copy()
    chunk_preds_list = []

    remaining = set(test_dates)
    for i in range(n_chunks):
        chunk_dates = set(test_dates[i * CHUNK_H : (i + 1) * CHUNK_H])
        preds       = nf.predict(df=history).reset_index(drop=True)
        preds['ds'] = pd.to_datetime(preds['ds'])
        matched     = preds[preds['ds'].isin(chunk_dates)]
        chunk_preds_list.append(matched)
        remaining  -= set(matched['ds'].unique())

        pred_rows = matched[['unique_id', 'ds', 'iTransformer']].rename(columns={'iTransformer': 'y'})
        pred_rows = add_calendar_features(pred_rows)
        history   = pd.concat([history, pred_rows], ignore_index=True).sort_values(['unique_id', 'ds']).reset_index(drop=True)

        if not remaining:
            break

    cluster_preds = pd.concat(chunk_preds_list, ignore_index=True)
    all_preds_itransformer.append(cluster_preds[['unique_id', 'ds', 'iTransformer']])

itransformer_preds = pd.concat(all_preds_itransformer, ignore_index=True)
mape_itransformer  = compute_per_user_period_mape(itransformer_preds, 'iTransformer', 'iTransformer')
print('\niTransformer done. Sample:')
print(mape_itransformer.groupby('period_id')['MAPE'].mean().round(2))

2026-03-24 18:00:39,948	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-03-24 18:00:40,053	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
Seed set to 42


Rolling prediction: 9 chunks of up to 720 hours each

--- iTransformer: cluster_0 (84 users) ---


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00,  6.45it/s]


Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


...

Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 116.35it/s]


Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 121.92it/s]


Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 121.30it/s]

iTransformer done. Sample:
period_id
1    12.07
2    15.65
3    16.88
4    29.74
Name: MAPE, dtype: float64


## 10. Combine & Save

In [11]:
all_mape = pd.concat([
    mape_autoarima,
    mape_autoets,
    mape_sarimax,
    mape_prophet,
    mape_itransformer,
], ignore_index=True)

all_mape = all_mape.dropna(subset=['MAPE']).reset_index(drop=True)
all_mape['MAPE'] = all_mape['MAPE'].round(4)

out_path = ARTIFACTS / 'per_user_period_mape.parquet'
all_mape.to_parquet(out_path, index=False)
print(f'Saved {len(all_mape)} rows -> {out_path}')
print(all_mape.head(10).to_string(index=False))

Saved 3120 rows -> /Users/jackchiu/assignments/Forecasting/IEOR4578_Electricity_Project/artifacts/clustering/per_user_period_mape.parquet
user_id     model  period_id    MAPE
 MT_124 AutoARIMA          1 15.8894
 MT_132 AutoARIMA          1 44.4087
 MT_156 AutoARIMA          1 24.4172
 MT_158 AutoARIMA          1 56.1096
 MT_159 AutoARIMA          1 76.5969
 MT_161 AutoARIMA          1 18.7433
 MT_162 AutoARIMA          1 64.2515
 MT_163 AutoARIMA          1 14.5941
 MT_166 AutoARIMA          1 12.9881
 MT_168 AutoARIMA          1  9.8573


## 11. Summary Table

In [12]:
summary = (
    all_mape
    .groupby(['model', 'period_id'])['MAPE']
    .mean()
    .round(2)
    .unstack('period_id')
)
summary.columns = [f'Period {c}' for c in summary.columns]
summary['Overall'] = all_mape.groupby('model')['MAPE'].mean().round(2)
print(summary.to_string())

              Period 1  Period 2  Period 3  Period 4  Overall
model                                                        
AutoARIMA        13.65     15.92     14.86     16.22    15.16
AutoETS          21.18     31.78     38.71     52.82    36.12
Prophet          16.52     22.02     18.42     22.26    19.80
SARIMAX          14.64     18.34     18.38     19.40    17.69
iTransformer     12.07     15.65     16.88     29.74    18.58


## 12. Cluster-Period MAPE Table

Aggregate per-user MAPEs to cluster level (mean across all users in each cluster).  
Saved as `cluster_period_mape.parquet` — loaded directly by the dashboard agent for the cluster MAPE metric card.  
60 rows: 3 clusters × 5 models × 4 periods.

In [13]:
user_to_cluster = {}
for _, row in mapping.iterrows():
    cid = row['cluster_id']
    key = 'outliers' if cid == -1 else f'cluster_{int(cid)}'
    user_to_cluster[row['user_id']] = key

cluster_mape_df = all_mape.copy()
cluster_mape_df['cluster_key'] = cluster_mape_df['user_id'].map(user_to_cluster)

cluster_period_mape = (
    cluster_mape_df
    .groupby(['cluster_key', 'model', 'period_id'])['MAPE']
    .mean()
    .round(4)
    .reset_index()
    .rename(columns={'MAPE': 'cluster_MAPE'})
)

out_path = ARTIFACTS / 'cluster_period_mape.parquet'
cluster_period_mape.to_parquet(out_path, index=False)
print(f'Saved {len(cluster_period_mape)} rows -> {out_path}')
print(cluster_period_mape.to_string(index=False))

Saved 60 rows -> /Users/jackchiu/assignments/Forecasting/IEOR4578_Electricity_Project/artifacts/clustering/cluster_period_mape.parquet
cluster_key        model  period_id  cluster_MAPE
  cluster_0    AutoARIMA          1       12.5059
  cluster_0    AutoARIMA          2       15.9818
  cluster_0    AutoARIMA          3       15.0187
  cluster_0    AutoARIMA          4       15.0838
  cluster_0      AutoETS          1       12.1423
  cluster_0      AutoETS          2       14.9893
  cluster_0      AutoETS          3       14.2579
  cluster_0      AutoETS          4       15.6699
  cluster_0      Prophet          1       14.6928
  cluster_0      Prophet          2       21.8440
  cluster_0      Prophet          3       17.4423
  cluster_0      Prophet          4       20.7482
  cluster_0      SARIMAX          1       12.8282
  cluster_0      SARIMAX          2       16.4410
  cluster_0      SARIMAX          3       15.4225
  cluster_0      SARIMAX          4       15.0857
  cluster_0 iTr